# Script 2: Training
**Dilated ResNet + Temporal Attention | 5-Fold Patient-Level Cross-Validation**
Master's Thesis: Deep Learning for Predicting Atrial Fibrillation

1. Run **Script 1** first — this notebook reads its Step 4 output (`DATA_ROOT/outputs/preprocessing/train_test_split.json`).
2. Select a GPU runtime (*Runtime → Change runtime type*).
3. Set `DATA_ROOT` in the **USER SETTING** cell to the same SHDB-AF folder used in Script 1.
4. Run all cells top to bottom (*Runtime → Run all*).

Outputs are written to `DATA_ROOT/outputs/results/dilated_resnet_attention/`: one `model_fold_{k}.keras` per fold and `cv_metadata.json`, which are used by Script 3. The preprocessed windows are cached in `DATA_ROOT/outputs/cache/`.

In [ ]:
# -*- coding: utf-8 -*-
# =============================================================================
# Script 2: Training
# Dilated ResNet + Temporal Attention  |  5-Fold Patient-Level Cross-Validation
# Master's Thesis: Deep Learning for Predicting Atrial Fibrillation
# =============================================================================
#
# PURPOSE:
#   Trains the AF prediction model on the 40-patient SHDB-AF window index
#   produced by Script 1, using stratified, patient-level K-fold
#   cross-validation. For every fold the script:
#
#   1. Holds out an inner validation set of patients (3 PAF + 3 non-AF) from
#      the fold's training patients. It drives early stopping and learning-rate
#      scheduling, so the fold's test patients are never used for training or
#      model selection.
#   2. Trains the Dilated ResNet + Temporal Attention model with balanced class
#      weights and random amplitude-scaling augmentation.
#   3. Saves the model with the best-validation-AUC weights restored, and
#      records the fold metadata (patient splits, restored epoch, history).
#
#   This script performs training only. Evaluation, metrics and plots are
#   produced by Script 3 from the saved models and cv_metadata.json.
#
# INPUT:
#   outputs/preprocessing/train_test_split.json
#       Window index from Script 1 (Step 4). Its train and test sets are
#       merged into one 40-patient pool, which is then re-split by the
#       cross-validation folds.
#   SHDB-AF WFDB records (.dat/.hea) in DATA_ROOT
#       Raw ECG signals (the same folder Script 1 reads the annotations from).
#
# PREPROCESSING (applied once, then cached to disk):
#   - Lead 0 only; 60 s windows at 200 Hz (12,000 samples per window).
#   - Per-patient amplitude clipping at the 0.5th / 99.5th percentiles.
#   - Per-window Z-score normalisation.
#
# OUTPUT FILES (written under DATA_ROOT/outputs, next to the Script 1 outputs):
#   cache/preprocessed_dataset_all_40.pkl
#       Preprocessed windows (reused on later runs).
#   results/<experiment>/model_fold_{k}.keras
#       One trained model per fold (k = 0 … n_folds-1).
#   results/<experiment>/cv_metadata.json
#       CONFIG + per-fold train/test indices, inner-validation patients,
#       restored epoch, best val AUC and training history.
#
#   <experiment> = EXPERIMENT_NAME ("dilated_resnet_attention" by default)
#
# DEPENDENCIES:
#   tensorflow (>= 2.11 for AdamW), scikit-learn (>= 1.0), wfdb, numpy
#   (+ standard library).
#
# ENVIRONMENT:
#   Google Colab with a GPU runtime, or Jupyter. The script uses notebook-only
#   syntax (`%pip install`), so it will not run as a plain `python script.py`.
#   Google Drive is mounted automatically when DATA_ROOT points to it
#   (/content/drive/...); otherwise DATA_ROOT can be any local folder.
#
# USAGE:
#   1. Run Script 1 to produce outputs/preprocessing/train_test_split.json.
#   2. Set DATA_ROOT in the USER SETTING cell to the same folder as in
#      Script 1. All other paths are derived from it; the CONFIGURATION
#      section holds the thesis hyperparameters.
#   3. Run the script top to bottom.
#   4. If the Script 1 output or the preprocessing settings change, delete the
#      cache file first; otherwise the cached dataset is loaded as is.
#
# =============================================================================

## USER SETTING

In [ ]:
# =============================================================================
# USER SETTING — the only line you need to edit
# =============================================================================
# Folder containing the SHDB-AF WFDB records (001.dat / 001.hea / 001.atr, ...)
# and AdditionalData.csv, exactly as downloaded from PhysioNet.
#   Colab example : "/content/drive/MyDrive/shdb-af/1.0.1"
#   Local example : "C:/data/shdb-af/1.0.1"  or  "/home/<user>/data/shdb-af/1.0.1"
#
# Every output of Scripts 1–3 is written under DATA_ROOT/outputs/.

DATA_ROOT = "/path/to/shdb-af/1.0.1"


## SETUP: Drive, Dependencies, Imports

In [ ]:
# 1. Mount Google Drive when DATA_ROOT points to it (Colab only)
if DATA_ROOT.startswith("/content/drive"):
    from google.colab import drive
    drive.mount('/content/drive')

# 2. Install the medical waveform library used to read the SHDB-AF records
%pip install wfdb --quiet

import datetime
import gc
import json
import os
import pickle
import random
import time
from collections import defaultdict

import numpy as np
import wfdb
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from sklearn.utils.class_weight import compute_class_weight

try:
    from sklearn.model_selection import StratifiedGroupKFold
except ImportError:
    raise ImportError(
        "StratifiedGroupKFold requires scikit-learn >= 1.0. "
        "Upgrade with: pip install -U scikit-learn"
    )

## CONFIGURATION

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================
# The file paths are derived from DATA_ROOT (USER SETTING cell). The settings
# below reproduce the thesis results. The full CONFIG dictionary is saved to
# cv_metadata.json, so every trained model can be traced back to the exact
# settings that produced it.

# ── Paths (derived from DATA_ROOT; identical in Scripts 1–3, do not edit) ─────
CSV_PATH            = os.path.join(DATA_ROOT, "AdditionalData.csv")
OUTPUT_ROOT         = os.path.join(DATA_ROOT, "outputs")
PREPROC_DIR         = os.path.join(OUTPUT_ROOT, "preprocessing")      # Script 1 outputs
SPLIT_FILE          = os.path.join(PREPROC_DIR, "train_test_split.json")
SECONDARY_TEST_FILE = os.path.join(PREPROC_DIR, "secondary_test_set.json")
CACHE_DIR           = os.path.join(OUTPUT_ROOT, "cache")              # Script 2 cache
EXPERIMENT_NAME     = "dilated_resnet_attention"
RESULTS_DIR         = os.path.join(OUTPUT_ROOT, "results", EXPERIMENT_NAME)   # Scripts 2–3

# ── Path checks — stop early with a clear message if a path is wrong ──────────
if not os.path.isdir(DATA_ROOT):
    raise FileNotFoundError(
        f"DATA_ROOT not found: {DATA_ROOT}\n"
        "Set DATA_ROOT in the USER SETTING cell to the SHDB-AF folder."
    )
if not any(f.endswith(".hea") for f in os.listdir(DATA_ROOT)):
    raise FileNotFoundError(f"No .hea record files found in DATA_ROOT: {DATA_ROOT}")
if not os.path.isfile(SPLIT_FILE):
    raise FileNotFoundError(f"Train/test split not found: {SPLIT_FILE}\nRun Script 1 first.")

CONFIG = {

    # ── Input paths (Script 1 outputs + raw records) ───────────────────────
    "data_dir":   DATA_ROOT,
    "json_path":  SPLIT_FILE,

    # SHDB-AF clinical metadata CSV (recorded in cv_metadata.json; not read here)
    "additional_data_path": CSV_PATH,

    # ── Output paths ───────────────────────────────────────────────────────
    "output_dir":     RESULTS_DIR,
    "cache_dir":      CACHE_DIR,
    "cache_filename": "preprocessed_dataset_all_40.pkl",

    # ── Labels (af_type value in the Script 1 JSON → class index) ──────────
    "label_map": {'PAF': 1, 'non-AF': 0},

    # ── Signal and windowing (must match Script 1) ─────────────────────────
    "record_prefix":          "",      # prefix before the zero-padded record ID
    "sampling_rate":          200,     # Hz
    "segment_length_sec":     60,      # window duration D used in Script 1
    "segment_length_samples": 12000,   # segment_length_sec × sampling_rate
    "leads":                  [0],     # ECG channel(s) used as model input

    # ── Preprocessing ──────────────────────────────────────────────────────
    "apply_clipping":       True,          # per-patient amplitude clipping
    "clip_method":          "percentile",
    "clip_percentile_low":  0.5,
    "clip_percentile_high": 99.5,
    "apply_normalization":  True,          # per-window Z-score normalisation
    "apply_augmentation":   True,          # random amplitude scaling (training set only)

    # ── Optimiser (AdamW) ──────────────────────────────────────────────────
    "learning_rate": 0.0001,
    "weight_decay":  0.00001,
    "beta_1":        0.9,
    "beta_2":        0.999,
    "epsilon":       1e-8,
    "amsgrad":       True,

    # ── Training ───────────────────────────────────────────────────────────
    "batch_size":            64,
    "max_epochs":            25,
    "paf_weight_multiplier": 1,   # extra factor on the balanced PAF class weight (1 = none)
    "early_stop_patience":   9,   # epochs without val_auc improvement before stopping
    "lr_reduce_patience":    5,   # epochs without val_loss improvement before halving the LR

    # ── Cross-validation ───────────────────────────────────────────────────
    "n_folds":                      5,
    "inner_val_patients_per_class": 3,   # patients per class held out for validation
    "random_seed":                  42,
}

os.makedirs(CONFIG["output_dir"], exist_ok=True)
os.makedirs(CONFIG["cache_dir"],  exist_ok=True)


def set_all_seeds(seed: int) -> None:
    """Fix the Python, NumPy and TensorFlow random seeds for reproducibility."""
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


set_all_seeds(CONFIG["random_seed"])

## DATA LOADING & PREPROCESSING

In [ ]:
# =============================================================================
# DATA LOADING & PREPROCESSING
# =============================================================================
# The ECG signal of every window in the Script 1 index is read from its WFDB
# record, clipped and normalised, and the result is cached to disk so later
# runs skip this step entirely.


def load_and_merge_step4_json(config: dict) -> dict:
    """
    Load the Script 1 (Step 4) window index and merge its train and test sets
    into a single patient pool. The patient-level split used for training is
    created afterwards by the cross-validation folds.
    """
    with open(config["json_path"], 'r') as f:
        data = json.load(f)
    merged_patients = {**data['train'], **data['test']}
    print(f"\n[DATA] Merged train and test sets. Total patients: {len(merged_patients)}")
    return merged_patients


def load_merged_dataset(merged_data: dict, config: dict) -> dict:
    """
    Read the ECG signal of every window in the merged index.

    Each window is sliced from the patient's selected WFDB record using its
    start_sec / end_sec. NaN samples are replaced with 0.

    Returns
    -------
    dict of numpy arrays, one entry per window:
        segments    — (n_windows, segment_length_samples, n_leads), float16
        labels      — (n_windows,), 1 = PAF, 0 = non-AF
        patient_ids — (n_windows,), patient ID
        start_secs  — (n_windows,), window start time in the recording (s)
    """
    data_dir  = config["data_dir"]
    prefix    = config["record_prefix"]
    leads     = config["leads"]
    fs        = config["sampling_rate"]
    label_map = config["label_map"]
    n_pats    = len(merged_data)

    all_segments, all_labels, all_patient_ids, all_start_secs = [], [], [], []

    print(f"\n[LOAD] Extracting WFDB signals for {n_pats} patients...")
    for pat_idx, (pat_id, pat_entry) in enumerate(merged_data.items()):
        label       = label_map[pat_entry['af_type']]
        record_path = os.path.join(data_dir, f"{prefix}{pat_entry['recording_id'].zfill(3)}")
        record      = wfdb.rdrecord(record_path, channels=leads)
        full_signal = record.p_signal

        for win in pat_entry['windows']:
            start      = int(round(win['start_sec'] * fs))
            end        = int(round(win['end_sec']   * fs))
            window_sig = full_signal[start:end, :]
            if np.isnan(window_sig).any():
                window_sig = np.nan_to_num(window_sig, nan=0.0)
            all_segments.append(window_sig.astype(np.float16))
            all_labels.append(label)
            all_patient_ids.append(pat_id)
            all_start_secs.append(win['start_sec'])

        if (pat_idx + 1) % 10 == 0:
            print(f"  Loaded {pat_idx + 1}/{n_pats} patients...")

    return {
        'segments':    np.array(all_segments,   dtype=np.float16),
        'labels':      np.array(all_labels,     dtype=np.int32),
        'patient_ids': np.array(all_patient_ids),
        'start_secs':  np.array(all_start_secs, dtype=np.float32),
    }


def verify_temporal_ordering(dataset: dict) -> None:
    """Raise an error if any patient's windows are not in strictly ascending time order."""
    print("\n[VERIFY] Checking temporal ordering of windows per patient...")
    groups = defaultdict(list)
    for pat, t in zip(dataset['patient_ids'], dataset['start_secs']):
        groups[pat].append(t)

    violations = 0
    for pat, times in groups.items():
        for i in range(1, len(times)):
            if times[i] <= times[i - 1]:
                print(f"  [FAIL] Patient {pat}: window {i} (t={times[i]:.2f}s) "
                      f"<= window {i-1} (t={times[i-1]:.2f}s).")
                violations += 1

    if violations == 0:
        print(f"  [PASS] All {len(groups)} patients are strictly time-ordered.")
    else:
        raise RuntimeError(f"Temporal ordering FAILED: {violations} violation(s).")


def preprocess_dataset(dataset: dict, config: dict) -> dict:
    """
    Per-patient amplitude clipping followed by per-window Z-score normalisation.

    Clipping — the low/high percentiles are computed per lead over ALL windows
               of a patient, and every sample is clipped to that range. This
               limits the influence of motion artefacts and spikes.
    Z-score  — each window is standardised per lead to zero mean and unit
               variance. Flat windows (std < 1e-8) are only mean-centred.

    Patients are processed one at a time to keep peak memory low. The
    segments are returned as float32.
    """
    print("\n[PREPROCESS] Clipping and Z-score normalising (per patient)...")
    segments    = dataset['segments']
    patient_ids = dataset['patient_ids']
    processed   = np.empty_like(segments, dtype=np.float32)

    for pat in np.unique(patient_ids):
        mask     = (patient_ids == pat)
        pat_data = segments[mask].astype(np.float32)

        if config['apply_clipping']:
            lower = np.percentile(pat_data, config['clip_percentile_low'],  axis=(0, 1))
            upper = np.percentile(pat_data, config['clip_percentile_high'], axis=(0, 1))
            for lead in range(pat_data.shape[-1]):
                if (upper[lead] - lower[lead]) > 1e-8:
                    pat_data[:, :, lead] = np.clip(pat_data[:, :, lead], lower[lead], upper[lead])

        if config['apply_normalization']:
            mu    = np.mean(pat_data, axis=1, keepdims=True)
            sigma = np.std( pat_data, axis=1, keepdims=True)
            sigma[sigma < 1e-8] = 1.0
            pat_data = (pat_data - mu) / sigma

        processed[mask] = pat_data
        del pat_data
        gc.collect()

    dataset['segments'] = processed
    del segments
    gc.collect()
    print("  [PREPROCESS] Complete.")
    return dataset


def prepare_and_cache_data(config: dict) -> dict:
    """
    Return the preprocessed dataset, loading it from the cache when available.

    Without a cache, the full pipeline runs (load index → read WFDB signals →
    verify ordering → preprocess) and the result is pickled to cache_dir.
    """
    cache_path = os.path.join(config['cache_dir'], config['cache_filename'])

    if os.path.exists(cache_path):
        print(f"\n[DATA CACHE] Found preprocessed data at {cache_path}")
        print("[DATA CACHE] Loading directly (skipping WFDB reading and preprocessing)...")
        with open(cache_path, 'rb') as f:
            dataset = pickle.load(f)
        return dataset

    print("\n[DATA CACHE] No cache found. Running full extraction and preprocessing...")
    merged_data = load_and_merge_step4_json(config)
    dataset     = load_merged_dataset(merged_data, config)
    verify_temporal_ordering(dataset)
    dataset     = preprocess_dataset(dataset, config)

    print(f"\n[DATA CACHE] Saving to {cache_path}...")
    with open(cache_path, 'wb') as f:
        pickle.dump(dataset, f, protocol=5)
    return dataset

## CROSS-VALIDATION SPLITS

In [ ]:
# =============================================================================
# CROSS-VALIDATION SPLITS — Stratified, Patient-Level
# =============================================================================

def create_cv_splits(dataset: dict, config: dict) -> list:
    """
    Build stratified, patient-level K-fold splits with StratifiedGroupKFold.

    Grouping by patient guarantees that no patient contributes windows to both
    the training and the test side of a fold; stratifying by label keeps the
    PAF / non-AF patient ratio balanced across folds. The split is computed on
    the unique patients (one label each) and then mapped back to window
    indices.

    Returns
    -------
    list of dicts, one per fold: {fold_number, train_indices, test_indices},
    where the indices refer to windows in `dataset`.
    """
    patient_ids_arr = dataset['patient_ids']
    labels_arr      = dataset['labels']
    unique_pats     = sorted(set(patient_ids_arr))
    pat_label       = {p: l for p, l in zip(patient_ids_arr, labels_arr)}

    pat_arr = np.array(unique_pats)
    lbl_arr = np.array([pat_label[p] for p in unique_pats])

    sgkf  = StratifiedGroupKFold(n_splits=config['n_folds'])
    folds = []

    print(f"\n[CV] Creating stratified {config['n_folds']}-fold patient-level splits")
    for fold_num, (train_idx, test_idx) in enumerate(
        sgkf.split(pat_arr, lbl_arr, groups=pat_arr)
    ):
        train_pats = set(pat_arr[train_idx])
        test_pats  = set(pat_arr[test_idx])

        assert len(train_pats & test_pats) == 0, "DATA LEAKAGE DETECTED!"

        test_labels = [pat_label[p] for p in test_pats]
        n_paf_test  = sum(test_labels)
        n_naf_test  = len(test_labels) - n_paf_test
        print(f"  Fold {fold_num}: test={len(test_pats)} patients "
              f"({n_paf_test} PAF + {n_naf_test} non-AF)")

        tr_idx = [int(i) for i, p in enumerate(patient_ids_arr) if p in train_pats]
        te_idx = [int(i) for i, p in enumerate(patient_ids_arr) if p in test_pats]

        folds.append({
            'fold_number':   fold_num,
            'train_indices': tr_idx,
            'test_indices':  te_idx,
        })
    return folds

## DATA AUGMENTATION

In [ ]:
# =============================================================================
# DATA AUGMENTATION
# =============================================================================

@tf.function
def tf_augment_ecg_window(segment, label):
    """
    Random amplitude scaling: with probability 0.5 the whole window is
    multiplied by a factor drawn uniformly from [0.85, 1.15].
    Applied on the fly to the training set only.
    """
    seg = tf.identity(segment)

    if tf.random.uniform([]) < 0.5:
        scale = tf.random.uniform([], minval=0.85, maxval=1.15)
        seg = seg * scale

    return seg, label

## MODEL ARCHITECTURE

In [ ]:
# =============================================================================
# MODEL ARCHITECTURE — Dilated ResNet + Temporal Attention
# =============================================================================

@keras.utils.register_keras_serializable(package="af_attention")
class TemporalAttentionSum(layers.Layer):
    """
    Sum the attention-weighted features over the temporal axis (axis=1).

    Implemented as a registered Keras layer (rather than a Lambda) so that
    saved models reload reliably. This class must be defined before loading a
    model with keras.models.load_model.
    """
    def call(self, inputs):
        return tf.reduce_sum(inputs, axis=1)

    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[-1])


def squeeze_and_excitation_1d(inputs, ratio=8, name_prefix=''):
    """1D Squeeze-and-Excitation block: learns a per-channel weighting of the features."""
    filters = inputs.shape[-1]
    se = layers.GlobalAveragePooling1D(name=f'{name_prefix}_se_gap')(inputs)
    se = layers.Dense(filters // ratio, activation='relu', name=f'{name_prefix}_se_dense1')(se)
    se = layers.Dense(filters, activation='sigmoid', name=f'{name_prefix}_se_dense2')(se)
    se = layers.Reshape((1, filters), name=f'{name_prefix}_se_reshape')(se)
    return layers.Multiply(name=f'{name_prefix}_se_mult')([inputs, se])


def dilated_resnet_block(inputs, filters, kernel_size, dilation_rates, pool_size, name_prefix):
    """
    Residual block of two dilated convolutions followed by Squeeze-and-Excitation.

    Conv(d1) → BN → ReLU → SpatialDropout → Conv(d2) → BN → SE, added to the
    shortcut (1×1 conv when the channel count changes), then ReLU and optional
    max pooling.
    """
    l2_reg = regularizers.l2(1e-4)

    x = layers.Conv1D(filters, kernel_size=kernel_size, padding='same',
                      dilation_rate=dilation_rates[0],
                      kernel_regularizer=l2_reg, name=f'{name_prefix}_conv1')(inputs)
    x = layers.BatchNormalization(name=f'{name_prefix}_bn1')(x)
    x = layers.ReLU(name=f'{name_prefix}_relu1')(x)
    x = layers.SpatialDropout1D(0.2, name=f'{name_prefix}_sdrop')(x)
    x = layers.Conv1D(filters, kernel_size=kernel_size, padding='same',
                      dilation_rate=dilation_rates[1],
                      kernel_regularizer=l2_reg, name=f'{name_prefix}_conv2')(x)
    x = layers.BatchNormalization(name=f'{name_prefix}_bn2')(x)
    x = squeeze_and_excitation_1d(x, ratio=8, name_prefix=name_prefix)

    if inputs.shape[-1] != filters:
        shortcut = layers.Conv1D(filters, kernel_size=1, padding='same',
                                 name=f'{name_prefix}_skip_conv')(inputs)
    else:
        shortcut = inputs

    x = layers.Add(name=f'{name_prefix}_add')([shortcut, x])
    x = layers.ReLU(name=f'{name_prefix}_relu2')(x)

    if pool_size > 1:
        x = layers.MaxPooling1D(pool_size=pool_size, strides=pool_size,
                                name=f'{name_prefix}_maxpool')(x)
    return x


def build_rse_model(config: dict) -> keras.Model:
    """
    Build the AF predictor: Dilated ResNet feature extractor + Temporal Attention.

    Input (segment_length_samples, n_leads)
      → Stem:      Conv1D(32, k=15, stride 2) → BN → ReLU → MaxPool(2)
      → Stage 1:   dilated residual block, 64 filters,  dilation (1, 2)
      → Stage 2:   dilated residual block, 128 filters, dilation (2, 4), MaxPool(2)
      → Stage 3:   dilated residual block, 128 filters, dilation (4, 8)
      → Attention: per-timestep score → softmax over time → weighted sum
      → Head:      Dense(64, 'fc_dense' = embedding layer) → Dropout(0.4)
                   → Dense(1, sigmoid) = P(PAF)
    """
    n_leads = len(config['leads'])
    seg_len = config['segment_length_samples']

    inputs = keras.Input(shape=(seg_len, n_leads), name='ecg_input')
    l2_reg = regularizers.l2(1e-4)

    # Stem
    x = layers.Conv1D(32, kernel_size=15, strides=2, padding='same',
                      kernel_regularizer=l2_reg, name='stem_conv')(inputs)
    x = layers.BatchNormalization(name='stem_bn')(x)
    x = layers.ReLU(name='stem_relu')(x)
    x = layers.MaxPooling1D(pool_size=2, strides=2, name='stem_maxpool')(x)

    # Dilated ResNet core
    x = dilated_resnet_block(x, filters=64, kernel_size=7,
                             dilation_rates=[1, 2], pool_size=1, name_prefix='stage1')
    x = dilated_resnet_block(x, filters=128, kernel_size=7,
                             dilation_rates=[2, 4], pool_size=2, name_prefix='stage2')
    x = dilated_resnet_block(x, filters=128, kernel_size=7,
                             dilation_rates=[4, 8], pool_size=1, name_prefix='stage3')

    # Temporal attention pooling
    attention_scores  = layers.Dense(64, activation='tanh', name='attn_dense1')(x)
    attention_scores  = layers.Dense(1, activation='linear', name='attn_dense2')(attention_scores)
    attention_weights = layers.Softmax(axis=1, name='attn_softmax')(attention_scores)
    attended_features = layers.Multiply(name='attn_multiply')([x, attention_weights])
    x = TemporalAttentionSum(name='attn_sum')(attended_features)

    # Classification head
    x = layers.Dense(64, activation='relu', kernel_regularizer=l2_reg, name='fc_dense')(x)
    x = layers.Dropout(0.4, name='fc_dropout')(x)
    output = layers.Dense(1, activation='sigmoid',
                          bias_initializer=tf.keras.initializers.Constant(0.0),
                          name='output')(x)

    return keras.Model(inputs=inputs, outputs=output, name="AF_Predictor_Dilated_Attention")

## TRAINING

In [ ]:
# =============================================================================
# TRAINING
# =============================================================================

def split_inner_validation(fold: dict, dataset: dict, config: dict):
    """
    Hold out `inner_val_patients_per_class` PAF and non-AF patients from the
    fold's training patients as the inner validation set.

    Patients are drawn with a seeded RNG, so the selection is reproducible.

    Returns
    -------
    inner_train_idx : window indices used for training
    inner_val_idx   : window indices used for validation
    inner_val_pats  : set of held-out patient IDs
    """
    all_train_idx     = np.array(fold['train_indices'])
    pat_ids_train_all = dataset['patient_ids'][all_train_idx]
    labels_train_all  = dataset['labels'][all_train_idx]

    unique_train_pats = sorted(set(pat_ids_train_all))
    pat_label_map     = {p: int(l) for p, l in zip(pat_ids_train_all, labels_train_all)}

    paf_pats = [p for p in unique_train_pats if pat_label_map[p] == 1]
    naf_pats = [p for p in unique_train_pats if pat_label_map[p] == 0]

    rng   = random.Random(config['random_seed'])
    n_val = config['inner_val_patients_per_class']
    inner_val_pats = set()
    inner_val_pats.update(rng.sample(paf_pats, min(n_val, len(paf_pats))))
    inner_val_pats.update(rng.sample(naf_pats, min(n_val, len(naf_pats))))

    inner_train_idx = [i for i in fold['train_indices']
                       if dataset['patient_ids'][i] not in inner_val_pats]
    inner_val_idx   = [i for i in fold['train_indices']
                       if dataset['patient_ids'][i] in inner_val_pats]

    print(f"\n  [CV] Inner split — train patients: {len(unique_train_pats) - len(inner_val_pats)} | "
          f"val patients: {sorted(str(p) for p in inner_val_pats)}")

    return inner_train_idx, inner_val_idx, inner_val_pats


def build_optimizer(config: dict):
    """AdamW on TensorFlow >= 2.11; older versions fall back to Adam without weight decay."""
    tf_version = tuple(int(v) for v in tf.__version__.split('.')[:2])
    if tf_version >= (2, 11):
        return tf.keras.optimizers.AdamW(
            learning_rate=config['learning_rate'], weight_decay=config['weight_decay'],
            beta_1=config['beta_1'], beta_2=config['beta_2'],
            epsilon=config['epsilon'], amsgrad=config['amsgrad'],
        )
    return tf.keras.optimizers.Adam(
        learning_rate=config['learning_rate'],
        beta_1=config['beta_1'], beta_2=config['beta_2'], epsilon=config['epsilon'],
    )


def train_fold(fold: dict, dataset: dict, config: dict) -> dict:
    """
    Train the model on one CV fold, save it, and return the fold's metadata.

    EarlyStopping monitors val_auc with restore_best_weights=True, so the saved
    model holds the weights of the epoch with the highest validation AUC. The
    stored training history is truncated at that epoch.
    """
    fold_num = fold['fold_number']
    print(f"\n{'#'*60}\n# FOLD {fold_num} / {config['n_folds']-1}\n{'#'*60}")

    # ── Inner train / validation split ────────────────────────────────────
    inner_train_idx, inner_val_idx, inner_val_pats = split_inner_validation(fold, dataset, config)

    train_X = dataset['segments'][inner_train_idx]
    train_y = dataset['labels'][inner_train_idx].astype(np.float32)
    val_X   = dataset['segments'][inner_val_idx]
    val_y   = dataset['labels'][inner_val_idx].astype(np.float32)

    # ── Model ─────────────────────────────────────────────────────────────
    keras.backend.clear_session()
    model = build_rse_model(config)
    model.compile(optimizer=build_optimizer(config), loss='binary_crossentropy',
                  metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])

    # ── Callbacks ─────────────────────────────────────────────────────────
    lr_scheduler = keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', mode='min', factor=0.5,
        patience=config['lr_reduce_patience'], min_lr=1e-6, verbose=1,
    )
    early_stop = keras.callbacks.EarlyStopping(
        monitor='val_auc', mode='max', patience=config['early_stop_patience'],
        restore_best_weights=True, verbose=1,
    )

    # ── Class weights (balanced, computed on the inner training set) ──────
    cw      = compute_class_weight('balanced', classes=np.unique(train_y), y=train_y)
    cw_dict = {0: cw[0], 1: cw[1] * config['paf_weight_multiplier']}

    # ── tf.data pipelines ─────────────────────────────────────────────────
    ds_train = tf.data.Dataset.from_tensor_slices((train_X, train_y)).shuffle(len(train_X))
    if config['apply_augmentation']:
        ds_train = ds_train.map(tf_augment_ecg_window, num_parallel_calls=tf.data.AUTOTUNE)
    ds_train = ds_train.batch(config['batch_size']).prefetch(tf.data.AUTOTUNE)

    ds_val = (tf.data.Dataset.from_tensor_slices((val_X, val_y))
              .batch(config['batch_size']))

    # ── Fit ───────────────────────────────────────────────────────────────
    history = model.fit(
        ds_train,
        epochs=config['max_epochs'],
        validation_data=ds_val,
        class_weight=cw_dict,
        callbacks=[lr_scheduler, early_stop],
        verbose=1,
    )

    # ── Epoch whose weights were restored (highest val_auc, 1-indexed) ────
    val_auc_hist    = history.history['val_auc']
    restored_epoch  = int(np.argmax(val_auc_hist)) + 1
    total_epochs    = len(history.history['loss'])
    best_val_auc    = float(np.max(val_auc_hist))
    final_train_auc = float(history.history['auc'][-1])

    print(f"\n  [TRAINING] Ran {total_epochs} epochs; best weights restored from "
          f"epoch {restored_epoch} (val_auc={best_val_auc:.4f}).")

    # ── Save model ────────────────────────────────────────────────────────
    model_file = f"model_fold_{fold_num}.keras"
    model_path = os.path.join(config['output_dir'], model_file)
    model.save(model_path)
    print(f"  [SAVED] Model -> {model_path}")

    # ── Fold metadata (history truncated at the restored epoch) ───────────
    def _trunc(key):
        return [float(v) for v in history.history[key][:restored_epoch]]

    history_truncated = {
        key: _trunc(key)
        for key in ('loss', 'val_loss', 'accuracy', 'val_accuracy', 'auc', 'val_auc')
    }

    fold_meta = {
        'fold_number':        fold_num,
        'train_indices':      [int(i) for i in fold['train_indices']],
        'test_indices':       [int(i) for i in fold['test_indices']],
        'inner_val_patients': sorted(str(p) for p in inner_val_pats),
        'restored_epoch':     restored_epoch,
        'total_epochs':       total_epochs,
        'best_val_auc':       best_val_auc,
        'final_train_auc':    final_train_auc,
        'model_file':         model_file,
        'history_truncated':  history_truncated,
    }

    del train_X, train_y, val_X, val_y, ds_train, ds_val, model
    gc.collect()
    return fold_meta

## RUN — Data Preparation and CV Splits

In [ ]:
print("=" * 60)
print("Script 2: Training")
print(f"Dilated ResNet + Temporal Attention  |  {CONFIG['n_folds']}-Fold Stratified CV")
print("=" * 60)

start_time = time.perf_counter()
start_dt   = datetime.datetime.now()
print(f"Start: {start_dt.strftime('%Y-%m-%d %H:%M:%S')}")

dataset = prepare_and_cache_data(CONFIG)
folds   = create_cv_splits(dataset, CONFIG)

## RUN — Cross-Validation Training

In [ ]:
fold_metas = [train_fold(fold, dataset, CONFIG) for fold in folds]

cv_metadata = {
    'experiment':      "Dilated_ResNet_Attention_CV_Training",
    'experiment_date': start_dt.isoformat(),
    'n_folds':         CONFIG['n_folds'],
    'cache_file':      CONFIG['cache_filename'],
    'config':          CONFIG,
    'folds':           fold_metas,
}
meta_path = os.path.join(CONFIG['output_dir'], "cv_metadata.json")
with open(meta_path, 'w') as f:
    json.dump(cv_metadata, f, indent=2)
print(f"\n[SAVED] CV metadata -> {meta_path}")

elapsed = time.perf_counter() - start_time
print(f"\n[DONE] Models and cv_metadata.json saved to: {CONFIG['output_dir']}")
print(f"End: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total time: {elapsed:.1f}s ({elapsed / 60:.1f} min)")
print("\nNext step: run Script 3 to evaluate the saved models.")